<a href="https://colab.research.google.com/github/lalitha-0890/2520080063---GenAI/blob/main/GALE_Week_2_Practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GALE Week 2 Practical — Colab Ready

**LLM Provider Interface using Gemini and Ollama**

Run the cells in order. Before Cell 8, add `GEMINI_API_KEY` in Colab **Secrets** (key icon) if you plan to use Gemini. Ollama cells may require a Colab runtime that supports the server process and can take time to download the model.

For the interactive cell at the end, choose `gemini` or `ollama` and enter a prompt when asked.

In [ ]:
!mkdir -p week-2
%cd week-2
!pwd


In [ ]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
!ollama --version

In [ ]:
import subprocess, time, requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.ok:
            print("Server says:", r.text)
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("Server did not start. Re-run this cell.")


In [ ]:
!ollama pull llama3.2

In [ ]:
!ollama list

In [ ]:
!curl -s http://localhost:11434/api/generate -d '{"model": "llama3.2", "prompt": "Say hello in one short sentence.", "stream": false}'

In [ ]:
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")

with open(".env", "w") as f:
    f.write(f'GEMINI_API_KEY="{key}"\n')

print(".env created. Key length:", len(key))


In [ ]:
!ls -a

In [ ]:
%%writefile requirements.txt
python-dotenv
pytest
google-genai
requests


In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from google import genai
print("google-genai imported OK")


In [ ]:
%%writefile llm_client.py
from abc import ABC, abstractmethod
from google import genai
import requests


# Common interface: every provider must implement generate()
class LLMProvider(ABC):

    @abstractmethod
    def generate(self, prompt):
        pass


# Gemini cloud provider
class GeminiProvider(LLMProvider):

    def __init__(self, api_key, model="gemini-3.8-flash"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def generate(self, prompt):
        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt
        )
        return response.text


# Ollama local provider
class OllamaProvider(LLMProvider):

    def __init__(self, model="llama3.2"):
        self.model = model
        self.url = "http://localhost:11434/api/generate"

    def generate(self, prompt):
        response = requests.post(
            self.url,
            json={
                "model": self.model,
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )
        response.raise_for_status()
        return response.json()["response"]


# Common client: the rest of your program only talks to this class
class LLMClient:

    def __init__(self, provider):
        self.provider = provider

    def generate(self, prompt):
        return self.provider.generate(prompt)


In [ ]:
%%writefile main.py
import os
from dotenv import load_dotenv
from llm_client import (
    GeminiProvider,
    OllamaProvider,
    LLMClient
)

load_dotenv(override=True)

provider_name = input(
    "Choose provider (gemini/ollama): "
).strip().lower()

if provider_name == "gemini":
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("Gemini API key not found in .env")
    provider = GeminiProvider(api_key)

elif provider_name == "ollama":
    provider = OllamaProvider("llama3.2")

else:
    raise ValueError("Invalid provider selected")

client = LLMClient(provider)

prompt = input("Enter your question: ")
answer = client.generate(prompt)

print("\nLLM Response:")
print(answer)


In [ ]:
%%writefile test_client.py
import unittest
from llm_client import LLMClient


# Fake provider for testing
class FakeProvider:
    def generate(self, prompt):
        return "AI stands for Artificial Intelligence."


class TestLLMClient(unittest.TestCase):

    def test_generate(self):
        provider = FakeProvider()
        client = LLMClient(provider)
        result = client.generate("Explain AI")
        self.assertEqual(
            result,
            "AI stands for Artificial Intelligence."
        )


if __name__ == "__main__":
    unittest.main()


In [ ]:
!python -m pytest test_client.py -v

In [ ]:
%run main.py